In [1]:
!pip install pycrs

In [2]:
import ee
import geemap

In [3]:
ee.Authenticate()


True

In [4]:
ee.Initialize()


In [5]:
# Create interactive map
Map = geemap.Map()

# =============================================================================
# SECTION 2: Load AOI and points
# =============================================================================

# Load forest and non-forest points and boundary of Kailali district

In [6]:
aoi = ee.FeatureCollection('projects/primeval-rune-481716-s3/assets/kailaliboundary')
forest = ee.FeatureCollection('projects/primeval-rune-481716-s3/assets/Forest').map(lambda f: f.set('class', 1))
water = ee.FeatureCollection('projects/primeval-rune-481716-s3/assets/water').map(lambda f: f.set('class', 2))
settelment = ee.FeatureCollection('projects/primeval-rune-481716-s3/assets/urban').map(lambda f: f.set('class', 3))
farmland = ee.FeatureCollection('projects/primeval-rune-481716-s3/assets/farmlannd').map(lambda f: f.set('class', 4))
barren = ee.FeatureCollection('projects/primeval-rune-481716-s3/assets/Barren').map(lambda f: f.set('class', 5))
sand = ee.FeatureCollection('projects/primeval-rune-481716-s3/assets/sand').map(lambda f: f.set('class', 0))

# Center map and add layers 

In [7]:
Map.centerObject(aoi,10)
Map.addLayer(aoi, {'color': 'red'}, 'AOI')

create map

In [8]:
Map

Map(center=[28.741500246455114, 80.88116600288161], controls=(WidgetControl(options=['position', 'transparent_…

# =============================================================================
# SECTION 3: Load and Process Sentinel-2 Data
# =============================================================================
# Load Sentinel-2 Harmonized Surface Reflectance

In [9]:
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")

In [10]:
# Filter for Post-Event Image
filtered_s2_post = (s2
                    .filterBounds(aoi)
                    .filterDate('2025-10-01', '2025-12-31')
                    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)))

# Create median composite and clip to AOI
S2_Median_Post = filtered_s2_post.median().clip(aoi)

# Visualize RGB composite
visual_parameters = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000
}

Map.addLayer(S2_Median_Post, visual_parameters, 'S2_Median_Post')


# =============================================================================
# SECTION 4: Load Pre-Event Sentinel-2 Data
# =============================================================================

In [11]:
# Filter for Pre-Event Image
S2_Pre = (s2
          .filterBounds(aoi)
          .filterDate('2024-10-01', '2024-12-31')
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 1))
          .median()
          .clip(aoi))

print("Pre and Post event images loaded successfully")

Pre and Post event images loaded successfully


# add the abbreviation for the model

In [12]:

# NDVI — Normalized Difference Vegetation Index
ndvi = S2_Median_Post.normalizedDifference(['B8', 'B4']).rename('NDVI')

# EVI — Enhanced Vegetation Index
evi = S2_Median_Post.expression(
'2.5 * (NIR - R) / (NIR + 6 * R - 7.5 * B + 1)', {
    'NIR': S2_Median_Post.select('B8'),
    'R': S2_Median_Post.select('B4'),
    'B': S2_Median_Post.select('B2')
}).rename('EVI')

# GNDVI — Green Normalized Difference Vegetation
gndvi = S2_Median_Post.normalizedDifference(['B8', 'B3']).rename('GNDVI')

# NDWI — Normalized Difference Water Index
ndwi = S2_Median_Post.normalizedDifference(['B8', 'B11']).rename('NDWI')

# NBR — Normalized Burn Ratio
nbr = S2_Median_Post.normalizedDifference(['B8', 'B12']).rename('NBR')

# NDUI — Normalized Difference Urban Index
# Load VIIRS nighttime lights (as NTL proxy for discontinued DMSP-OLS)
viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG") \
    .filterBounds(aoi) \
    .filterDate('2025-10-01', '2025-12-31') \
    .select('avg_rad') \
    .median() \
    .clip(aoi)

# Normalize NTL to 0-1 scale (min-max normalization is common practice)
ntl_min = viirs.reduceRegion(ee.Reducer.min(), aoi, 500).get('avg_rad')
ntl_max = viirs.reduceRegion(ee.Reducer.max(), aoi, 500).get('avg_rad')
ntl_norm = viirs.subtract(ee.Number(ntl_min)).divide(
    ee.Number(ntl_max).subtract(ee.Number(ntl_min))
).rename('NTL')

# Calculate NDUI
ndui = ntl_norm.subtract(ndvi).divide(ntl_norm.add(ndvi)).rename('NDUI')

# =============================================================================
# SECTION 5: Call the abbreviation
# =============================================================================

In [13]:
Map

Map(center=[28.741500246455114, 80.88116600288161], controls=(WidgetControl(options=['position', 'transparent_…

# =============================================================================
# SECTION 6: Image for classification
# =============================================================================

In [14]:
 #Combine your raw bands + all calculated indices into ONE multi-band image
inputImage = ee.Image.cat([
    S2_Median_Post.select(['B2', 'B3', 'B4', 'B8']),  # raw bands
    ndvi,
    evi,
    gndvi,
    ndwi,
    nbr,
    ndui
]).clip(aoi)

# 2. Get the band names directly from the combined image (no arguments needed)
inputProperties = inputImage.bandNames()
print('Input bands:', inputProperties.getInfo())

Input bands: ['B2', 'B3', 'B4', 'B8', 'NDVI', 'EVI', 'GNDVI', 'NDWI', 'NBR', 'NDUI']


# =============================================================================
# SECTION 7: concatenate the location
# =============================================================================

In [15]:
sample = sand.merge(forest).merge(water).merge(barren).merge(settelment).merge(farmland)

# =============================================================================
# SECTION 9: split the training data into training and validation sets
# =============================================================================

In [16]:
sample = sample.randomColumn()
training_sample = sample.filter('random <= 0.8')
validation_sample = sample.filter('random > 0.8')

# =============================================================================
# SECTION 8: Train the model in a Random Forest classifier
# =============================================================================

In [27]:
trained = inputImage.sampleRegions(
  collection = sample,
  scale= 10,
  properties= ['class'],
  tileScale= 2
)

In [28]:
task = ee.batch.Export.table.toDrive(
    collection=trained,
    description='exported training data',
    fileFormat='CSV',
)
task.start()

# =============================================================================
# SECTION 10: Apply the classification
# =============================================================================

In [18]:
classifier = ee.Classifier.smileRandomForest(50).train(trained, 'class', inputProperties)

In [19]:
inputImage

# =============================================================================
# SECTION 11: visualize the map
# =============================================================================

In [20]:
value = [0, 1, 2, 3, 4, 5]
palette = ['A52A2A', '008000', '0000FF', 'FF0000', '00FF00', 'DEB887']
landCover = inputImage.classify(classifier).rename('LULC').set('LULC_class_palette', palette, 'LULC_class_values', value)
Map.addLayer(landCover, {}, 'LULC')

# =============================================================================
# SECTION 12: show the map
# =============================================================================

In [21]:
Map

Map(bottom=109506.0, center=[28.741500246455114, 80.88116600288161], controls=(WidgetControl(options=['positio…

map

# =============================================================================
# SECTION 13: find the accuracy of the model
# =============================================================================

In [22]:
display('Results of trained classifier', classifier.explain())
train_accuracy = classifier.confusionMatrix()
display('Training error matrix', train_accuracy)
display('Training overall accuracy', train_accuracy.accuracy())

'Results of trained classifier'

'Training error matrix'

'Training overall accuracy'

# =============================================================================
# SECTION 14: accuracy using the validation set
# =============================================================================

In [23]:
validation_sampled = inputImage.sampleRegions(
    collection = validation_sample,
    scale = 10,
    properties=['class'],
    tileScale=2
)

In [24]:
validated = validation_sampled.classify(classifier)
validation_accuracy = validated.errorMatrix('class', 'classification')
print('Validation error matrix:', validation_accuracy.getInfo())
print('Validation accuracy:', validation_accuracy.accuracy().getInfo())

Validation error matrix: [[7, 0, 0, 0, 3, 0], [0, 75, 0, 0, 0, 1], [3, 0, 15, 5, 0, 1], [0, 0, 1, 34, 2, 1], [0, 0, 0, 2, 15, 1], [3, 1, 1, 0, 0, 5]]
Validation accuracy: 0.8579545454545454
